## Setup: Directory Configuration and Library Imports


In [1]:
import pandas as pd
from pathlib import Path

In [2]:
# Configure paths
PREPROCESSED_DIR = Path('../../data/preprocessed')
MERGED_DIR = Path('../../data/merged')

# Create output directory if it doesn't exist
MERGED_DIR.mkdir(parents=True, exist_ok=True)

print(f"Preprocessed data location: {PREPROCESSED_DIR.resolve()}")
print(f"Output location: {MERGED_DIR.resolve()}")
print(f"Output directory created: {MERGED_DIR.exists()}")

Preprocessed data location: C:\Users\Dell\Documents\MSc AAI\FAIDM - WM9QG\Group Assessment\data-mining\data\preprocessed
Output location: C:\Users\Dell\Documents\MSc AAI\FAIDM - WM9QG\Group Assessment\data-mining\data\merged
Output directory created: True


## Step 1: Merging Demographics & Engagement Data

We integrate student demographic information, course registration details, StudentVle interaction metrics, and course metadata through a series of left joins starting with studentInfo as the base table. Finally integrating Vle for total possible interations.


### Loading Preprocessed Tables


In [24]:
# Load student information (demographics)
print("Loading preprocessed tables...")
student_info = pd.read_csv(PREPROCESSED_DIR / 'studentInfo.csv')
print(f"studentInfo: {student_info.shape}")
print(student_info.info())

# Load student registration data
student_reg = pd.read_csv(PREPROCESSED_DIR / 'studentRegistration.csv')
print(f"\nstudentRegistration: {student_reg.shape}")
print(student_reg.info())

Loading preprocessed tables...
studentInfo: (32593, 12)
<class 'pandas.DataFrame'>
RangeIndex: 32593 entries, 0 to 32592
Data columns (total 12 columns):
 #   Column                Non-Null Count  Dtype
---  ------                --------------  -----
 0   code_module           32593 non-null  str  
 1   code_presentation     32593 non-null  str  
 2   id_student            32593 non-null  int64
 3   gender                32593 non-null  str  
 4   region                32593 non-null  str  
 5   highest_education     32593 non-null  str  
 6   imd_band              32593 non-null  str  
 7   age_band              32593 non-null  str  
 8   num_of_prev_attempts  32593 non-null  int64
 9   studied_credits       32593 non-null  int64
 10  disability            32593 non-null  str  
 11  final_result          32593 non-null  str  
dtypes: int64(3), str(9)
memory usage: 3.0 MB
None

studentRegistration: (32593, 5)
<class 'pandas.DataFrame'>
RangeIndex: 32593 entries, 0 to 32592
Data column

In [25]:
# Load student VLE interaction data (aggregated)
student_vle = pd.read_csv(PREPROCESSED_DIR / 'studentVle.csv')
print(f"studentVle: {student_vle.shape}")
print(student_vle.info())

# Load course metadata
courses = pd.read_csv(PREPROCESSED_DIR / 'courses.csv')
print(f"\ncourses: {courses.shape}")
print(courses.info())

studentVle: (29228, 7)
<class 'pandas.DataFrame'>
RangeIndex: 29228 entries, 0 to 29227
Data columns (total 7 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   code_module         29228 non-null  str    
 1   code_presentation   29228 non-null  str    
 2   id_student          29228 non-null  int64  
 3   total_clicks        29228 non-null  int64  
 4   avg_clicks_per_day  29228 non-null  float64
 5   num_sites           29228 non-null  int64  
 6   num_unique_sites    29228 non-null  int64  
dtypes: float64(1), int64(4), str(2)
memory usage: 1.6 MB
None

courses: (22, 3)
<class 'pandas.DataFrame'>
RangeIndex: 22 entries, 0 to 21
Data columns (total 3 columns):
 #   Column                      Non-Null Count  Dtype
---  ------                      --------------  -----
 0   code_module                 22 non-null     str  
 1   code_presentation           22 non-null     str  
 2   module_presentation_length  22 non-null   

In [26]:
# Load course metadata
vle = pd.read_csv(PREPROCESSED_DIR / 'vle.csv')
print(f"\nvle: {vle.shape}")
print(vle.info())


vle: (22, 3)
<class 'pandas.DataFrame'>
RangeIndex: 22 entries, 0 to 21
Data columns (total 3 columns):
 #   Column                   Non-Null Count  Dtype
---  ------                   --------------  -----
 0   code_module              22 non-null     str  
 1   code_presentation        22 non-null     str  
 2   total_unique_activities  22 non-null     int64
dtypes: int64(1), str(2)
memory usage: 660.0 bytes
None


### Performing Left Joins

Starting with studentInfo, we perform sequential left joins to add registration data, VLE interactions, and course metadata.


In [27]:
# Initialize master dataframe with studentInfo
master = student_info.copy()
print(f"Initial master dataframe shape: {master.shape}")

# Define join keys
join_keys = ['id_student', 'code_module', 'code_presentation']
print(f"Join keys: {join_keys}")

Initial master dataframe shape: (32593, 12)
Join keys: ['id_student', 'code_module', 'code_presentation']


In [28]:
# Left join with student registration
print("Merging studentRegistration...")
master = master.merge(
    student_reg,
    on=join_keys,
    how='left',
    suffixes=('', '_reg')
)
print(f"After registration merge: {master.shape}")
print(f"Columns added: {[col for col in master.columns if 'date_' in col]}")
print(master.info())

Merging studentRegistration...
After registration merge: (32593, 14)
Columns added: ['date_registration', 'date_unregistration']
<class 'pandas.DataFrame'>
RangeIndex: 32593 entries, 0 to 32592
Data columns (total 14 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   code_module           32593 non-null  str    
 1   code_presentation     32593 non-null  str    
 2   id_student            32593 non-null  int64  
 3   gender                32593 non-null  str    
 4   region                32593 non-null  str    
 5   highest_education     32593 non-null  str    
 6   imd_band              32593 non-null  str    
 7   age_band              32593 non-null  str    
 8   num_of_prev_attempts  32593 non-null  int64  
 9   studied_credits       32593 non-null  int64  
 10  disability            32593 non-null  str    
 11  final_result          32593 non-null  str    
 12  date_registration     32548 non-null  float64
 13  dat

In [29]:
# Left join with student VLE data
print("Merging studentVle...")
master = master.merge(
    student_vle,
    on=join_keys,
    how='left',
    suffixes=('', '_vle')
)
print(f"After StudentVle merge: {master.shape}")
print(f"StudentVle columns: {[col for col in master.columns if 'click' in col or 'sites' in col]}")
print(master.isna().sum())

Merging studentVle...
After StudentVle merge: (32593, 18)
StudentVle columns: ['total_clicks', 'avg_clicks_per_day', 'num_sites', 'num_unique_sites']
code_module                 0
code_presentation           0
id_student                  0
gender                      0
region                      0
highest_education           0
imd_band                    0
age_band                    0
num_of_prev_attempts        0
studied_credits             0
disability                  0
final_result                0
date_registration          45
date_unregistration     22521
total_clicks             3365
avg_clicks_per_day       3365
num_sites                3365
num_unique_sites         3365
dtype: int64


In [30]:
# Left join with courses metadata
print("Merging courses...")
master = master.merge(
    courses[['code_module', 'code_presentation', 'module_presentation_length']],
    on=['code_module', 'code_presentation'],
    how='left'
)
print(f"After courses merge: {master.shape}")
print(f"Length column added: 'module_presentation_length' present = {'module_presentation_length' in master.columns}")
print(master.isna().sum())

Merging courses...
After courses merge: (32593, 19)
Length column added: 'module_presentation_length' present = True
code_module                       0
code_presentation                 0
id_student                        0
gender                            0
region                            0
highest_education                 0
imd_band                          0
age_band                          0
num_of_prev_attempts              0
studied_credits                   0
disability                        0
final_result                      0
date_registration                45
date_unregistration           22521
total_clicks                   3365
avg_clicks_per_day             3365
num_sites                      3365
num_unique_sites               3365
module_presentation_length        0
dtype: int64


In [31]:
# Left join with vle metadata
print("Merging vle...")
master = master.merge(
    vle[['code_module', 'code_presentation', 'total_unique_activities']],
    on=['code_module', 'code_presentation'],
    how='left'
)
print(f"After vle merge: {master.shape}")
print(f"Length column added: 'total_unique_activities' present = {'total_unique_activities' in master.columns}")
print(master.isna().sum())

Merging vle...
After vle merge: (32593, 20)
Length column added: 'total_unique_activities' present = True
code_module                       0
code_presentation                 0
id_student                        0
gender                            0
region                            0
highest_education                 0
imd_band                          0
age_band                          0
num_of_prev_attempts              0
studied_credits                   0
disability                        0
final_result                      0
date_registration                45
date_unregistration           22521
total_clicks                   3365
avg_clicks_per_day             3365
num_sites                      3365
num_unique_sites               3365
module_presentation_length        0
total_unique_activities           0
dtype: int64


### Validation: Missing Data Analysis


In [32]:
# Check for missing values
print("Missing values by column:")
missing_data = master.isnull().sum()
missing_pct = (master.isnull().sum() / len(master)) * 100
missing_df = pd.DataFrame({
    'Missing_Count': missing_data,
    'Missing_Percentage': missing_pct
}).sort_values('Missing_Count', ascending=False)

print(missing_df[missing_df['Missing_Count'] > 0])
print(f"\nTotal rows with complete data: {master.isnull().sum(axis=1).eq(0).sum()}")

Missing values by column:
                     Missing_Count  Missing_Percentage
date_unregistration          22521           69.097659
avg_clicks_per_day            3365           10.324303
num_sites                     3365           10.324303
total_clicks                  3365           10.324303
num_unique_sites              3365           10.324303
date_registration               45            0.138066

Total rows with complete data: 7095


### Ghost Student (with no date_registration)


In [33]:
# Create a dataframe of just the students missing a registration date
ghost_students = master[master['date_registration'].isnull()]

print(f"--- Investigation of {len(ghost_students)} students missing date_registration ---")

# 1. Did any of them withdraw?
withdrawn_ghosts = ghost_students['date_unregistration'].notnull().sum()
print(f"Students who 'un-registered' without a registration date: {withdrawn_ghosts}")

# 2. Did any of them actually use the VLE?
active_ghosts = (ghost_students['total_clicks'] > 0).sum()
print(f"Students who have VLE clicks but no registration date: {active_ghosts}")

# 3. What were their final results?
print("\nFinal Results of these 45 students:")
print(ghost_students['final_result'].value_counts())

# 4. Show a sample of these records
ghost_students[['id_student', 'code_module', 'code_presentation', 'final_result', 'total_clicks']].head()

--- Investigation of 45 students missing date_registration ---
Students who 'un-registered' without a registration date: 39
Students who have VLE clicks but no registration date: 7

Final Results of these 45 students:
final_result
Withdrawn    39
Fail          5
Pass          1
Name: count, dtype: int64


,id_student,code_module,code_presentation,final_result,total_clicks
2344,630346,BBB,2013B,Fail,NaN
2538,57369,BBB,2013J,Withdrawn,NaN
2759,342678,BBB,2013J,Withdrawn,NaN
5356,582496,BBB,2014B,Withdrawn,NaN
5490,607646,BBB,2014B,Withdrawn,NaN


### Handling Missing Data


1. Drop records with missing date_registration as it is non-negligible, might be fake and will hinder further merging and precision of model by feeding incorrect information.


In [34]:
master = master.dropna(subset=['date_registration'])
print(f"Dropped 45 rows. New shape: {master.shape}")

Dropped 45 rows. New shape: (32548, 20)


2. Fill missing VLE interaction metrics (total_clicks, avg_clicks_per_day, num_sites) with 0, as these represent students with no VLE activity.


In [35]:
# Identify studentVle columns to fill
studentVle_fill_cols = ['total_clicks', 'avg_clicks_per_day', 'num_sites', 'num_unique_sites']
studentVle_fill_cols_present = [col for col in studentVle_fill_cols if col in master.columns]

print(f"StudentVLE columns to fill with 0: {studentVle_fill_cols_present}")

# Fill missing studentVle metrics with 0 (no activity = 0 clicks)
# Use fillna without inplace to avoid Copy-on-Write warnings
for col in studentVle_fill_cols_present:
    before_fill = master[col].isnull().sum()
    master[col] = master[col].fillna(0)
    after_fill = master[col].isnull().sum()
    print(f"{col}: {before_fill} → {after_fill} missing values")

print("\nMissing values after studentVle imputation:")
print(master.isnull().sum()[master.isnull().sum() > 0])

StudentVLE columns to fill with 0: ['total_clicks', 'avg_clicks_per_day', 'num_sites', 'num_unique_sites']
total_clicks: 3327 → 0 missing values
avg_clicks_per_day: 3327 → 0 missing values
num_sites: 3327 → 0 missing values
num_unique_sites: 3327 → 0 missing values

Missing values after studentVle imputation:
date_unregistration    22515
dtype: int64


3. Drop date_unregistration. It is a feature that is filled after the student has withdrawn from the module-presentation. (Dropping to avoid data leakage)

In [36]:
# Drop date_unregistration as it's not needed
if 'date_unregistration' in master.columns:
    master = master.drop(columns=['date_unregistration'])
    print("\nDropped 'date_unregistration' column from master dataset")


Dropped 'date_unregistration' column from master dataset


## Step 2: Calculating Normalized Performance (Avoiding Data Leakage)

We calculate weighted coursework scores from assessments, excluding exams to avoid data leakage. This represents student performance on graded coursework only.


### Loading Assessment Data


In [39]:
# Load student assessment scores
student_assess = pd.read_csv(PREPROCESSED_DIR / 'studentAssessment.csv')
print(f"studentAssessment: {student_assess.shape}")
print(student_assess.head())
print(f"\nColumns: {student_assess.columns.tolist()}")

# Load assessment metadata (weights)
assessments = pd.read_csv(PREPROCESSED_DIR / 'assessments.csv')
print(f"\nassessments: {assessments.shape}")
print(assessments.head())
print(f"\nColumns: {assessments.columns.tolist()}")

studentAssessment: (173912, 5)
   id_assessment  id_student  date_submitted  is_banked  score
0           1752       11391              18          0   78.0
1           1752       28400              22          0   70.0
2           1752       31604              17          0   72.0
3           1752       32885              26          0   69.0
4           1752       38053              19          0   79.0

Columns: ['id_assessment', 'id_student', 'date_submitted', 'is_banked', 'score']

assessments: (206, 6)
  code_module code_presentation  id_assessment assessment_type   date  weight
0         AAA             2013J           1752             TMA   19.0    10.0
1         AAA             2013J           1753             TMA   54.0    20.0
2         AAA             2013J           1754             TMA  117.0    20.0
3         AAA             2013J           1755             TMA  166.0    20.0
4         AAA             2013J           1756             TMA  215.0    30.0

Columns: ['code_m

### Filtering Out Exams

Remove all exam records to focus on coursework (TMA/CMA) only, avoiding data leakage from future exam performance.


In [40]:
# Check assessment types
print("Assessment types in data:")
print(assessments['assessment_type'].value_counts())

# Filter out exams
assessments_no_exam = assessments[assessments['assessment_type'] != 'Exam'].copy()
# assessments_no_exam = assessments.copy()
print(f"\nAssessments after filtering exams: {assessments_no_exam.shape[0]} (removed {assessments.shape[0] - assessments_no_exam.shape[0]})")
print(f"Assessment types remaining: {assessments_no_exam['assessment_type'].unique()}")

Assessment types in data:
assessment_type
TMA     106
CMA      76
Exam     24
Name: count, dtype: int64

Assessments after filtering exams: 182 (removed 24)
Assessment types remaining: <StringArray>
['TMA', 'CMA']
Length: 2, dtype: str


### Merging Assessment Data with Weights


In [56]:
# Merge student assessments with assessment metadata to get weights
assessment_with_weights = student_assess.merge(
    assessments_no_exam[['code_module', 'code_presentation', 'id_assessment', 'assessment_type', 'weight', 'date']],
    on='id_assessment',
    how='inner'
)

print(f"Assessment records with weights: {assessment_with_weights.shape}")
print(assessment_with_weights.head())
print(f"\nColumns in assessment_with_weights: {assessment_with_weights.columns.tolist()}")

Assessment records with weights: (168953, 10)
   id_assessment  id_student  date_submitted  is_banked  score code_module  \
0           1752       11391              18          0   78.0         AAA   
1           1752       28400              22          0   70.0         AAA   
2           1752       31604              17          0   72.0         AAA   
3           1752       32885              26          0   69.0         AAA   
4           1752       38053              19          0   79.0         AAA   

  code_presentation assessment_type  weight  date  
0             2013J             TMA    10.0  19.0  
1             2013J             TMA    10.0  19.0  
2             2013J             TMA    10.0  19.0  
3             2013J             TMA    10.0  19.0  
4             2013J             TMA    10.0  19.0  

Columns in assessment_with_weights: ['id_assessment', 'id_student', 'date_submitted', 'is_banked', 'score', 'code_module', 'code_presentation', 'assessment_type', 'weight',

### Checking if tma_cma_weight_score was correct

The inner removes the Exam
We get weight_values < 100, because the Exam is not included

Suppose:

- TMA 40 10
- CMA 30 10
- TMA 100 0
- EXAM 70 80

The weight will be = (40 * 10 + 30*10 + 100 \* 0) / (10+10+0)

Suppose:
That student has not given one of the exams:

- TMA - 10
- CMA 30 10
- TMA 100 0
- EXAM 70 80

The weight will be = (30* 10 + 100*0) / (10+10+0)


#### Filter out exams

assessments_exam = assessments[assessments['assessment_type'] == 'Exam'].copy()

#### assessments_no_exam = assessments.copy()

print(f"\nAssessments after filtering exams: {assessments_exam.shape[0]} (removed {assessments.shape[0] - assessments_exam.shape[0]})")
print(f"Assessment types remaining: {assessments_exam['assessment_type'].unique()}")

#### Merge student assessments with assessment metadata to get weights

assessment_with_weights = student_assess.merge(
assessments_exam[['id_assessment', 'assessment_type', 'weight', 'code_module', 'code_presentation']],
on='id_assessment',
how='inner'
)

print(f"Assessment records with weights: {assessment_with_weights.shape}")
print(assessment_with_weights.head())
print(f"\nColumns in assessment_with_weights: {assessment_with_weights.columns.tolist()}")


#### Filter out exams

assessments_exam = assessments[assessments['assessment_type'] == 'Exam'].copy()

#### assessments_no_exam = assessments.copy()

print(f"\nAssessments after filtering exams: {assessments_exam.shape[0]} (removed {assessments.shape[0] - assessments_exam.shape[0]})")
print(f"Assessment types remaining: {assessments_exam['assessment_type'].unique()}")

#### Merge student assessments with assessment metadata to get weights

assessment_with_weights = student_assess.merge(
assessments_exam[['id_assessment', 'assessment_type', 'weight', 'code_module', 'code_presentation']],
on='id_assessment',
how='inner'
)

print(f"Assessment records with weights: {assessment_with_weights.shape}")
print(assessment_with_weights.head())
print(f"\nColumns in assessment_with_weights: {assessment_with_weights.columns.tolist()}")


### Calculating Weighted Scores


In [57]:
# Calculate weighted points for each assessment
assessment_with_weights['weighted_score'] = assessment_with_weights['score'] * assessment_with_weights['weight']

print("Sample weighted points calculation:")
print(assessment_with_weights.head(10))

Sample weighted points calculation:
   id_assessment  id_student  date_submitted  is_banked  score code_module  \
0           1752       11391              18          0   78.0         AAA   
1           1752       28400              22          0   70.0         AAA   
2           1752       31604              17          0   72.0         AAA   
3           1752       32885              26          0   69.0         AAA   
4           1752       38053              19          0   79.0         AAA   
5           1752       45462              20          0   70.0         AAA   
6           1752       45642              18          0   72.0         AAA   
7           1752       52130              19          0   72.0         AAA   
8           1752       53025               9          0   71.0         AAA   
9           1752       57506              18          0   68.0         AAA   

  code_presentation assessment_type  weight  date  weighted_score  
0             2013J             TMA  

### Aggregating to Student Level

Group by student-module-presentation combination and sum weighted points and total available weights.


Get the total available weights for the assessment for each module-presentation

In [83]:
# CORRECTED LOGIC: Account for missing assessments
# ====================================================

# Step 1: Calculate total possible weight for EACH module-presentation
# This ensures students who miss assessments get penalized appropriately
module_total_weights = assessments_no_exam.groupby(['code_module', 'code_presentation'])['weight'].sum().reset_index()
module_total_weights.rename(columns={'weight': 'total_weight'}, inplace=True)

print(f"Total possible weights per module-presentation: {module_total_weights.shape}")
print(module_total_weights.head(22))
print(f"\nWeight coverage statistics:")
print(module_total_weights['total_weight'].describe())

# Step 2: Get all students x module-presentation combinations from master
student_module_combos = master[join_keys].drop_duplicates().copy()
print(f"\nUnique student-module-presentation combinations: {student_module_combos.shape}")

# Step 3: Merge with module total weights (baseline: all students start with 0 weighted points)
student_performance = student_module_combos.merge(
    module_total_weights,
    on=['code_module', 'code_presentation'],
    how='left'
).copy()

Total possible weights per module-presentation: (22, 3)
   code_module code_presentation  total_weight
0          AAA             2013J         100.0
1          AAA             2014J         100.0
2          BBB             2013B         100.0
3          BBB             2013J         100.0
4          BBB             2014B         100.0
5          BBB             2014J         100.0
6          CCC             2014B         100.0
7          CCC             2014J         100.0
8          DDD             2013B         100.0
9          DDD             2013J         100.0
10         DDD             2014B         100.0
11         DDD             2014J         100.0
12         EEE             2013J         100.0
13         EEE             2014B         100.0
14         EEE             2014J         100.0
15         FFF             2013B         100.0
16         FFF             2013J         100.0
17         FFF             2014B         100.0
18         FFF             2014J         100.0
19  

Get all the submitted assessment and their sum_weighted_scores (assessments that students have submitted).

In [84]:
# Step 4: Add actual submitted scores where they exist
student_submitted = assessment_with_weights.groupby(join_keys).agg({
    'weighted_score': 'sum'
}).reset_index()

student_submitted.rename(columns={'weighted_score': 'sum_weighted_score'}, inplace=True)
print(f"\nStudent submitted assessments: {student_submitted.shape}")
print(f"\nStudent submitted assessments columns: {student_submitted.columns.tolist()}")

# Update with actual scores (merges in the submitted values)
student_performance = student_performance.merge(
    student_submitted,
    on=join_keys,
    how='left'
)
print(f"\nStudent performance after merging submitted scores: {student_performance.shape}")
print(student_performance.head())


Student submitted assessments: (25839, 4)

Student submitted assessments columns: ['id_student', 'code_module', 'code_presentation', 'sum_weighted_score']

Student performance after merging submitted scores: (32548, 5)
   id_student code_module code_presentation  total_weight  sum_weighted_score
0       11391         AAA             2013J         100.0              8240.0
1       28400         AAA             2013J         100.0              6540.0
2       30268         AAA             2013J         100.0                 NaN
3       31604         AAA             2013J         100.0              7630.0
4       32885         AAA             2013J         100.0              5500.0


Normalization: Computing Coursework Score (0-100 scale)

- Formula: `tma_cma_weighted_score = (sum_weight_score / total_weights)`

This scales the score to 0-100 based only on coursework completed, ignoring the 50% usually reserved for exams.


In [85]:
student_performance['tma_cma_weighted_score'] = student_performance['sum_weighted_score'] / student_performance['total_weight']

print(f"\nStudent performance aggregated: {student_performance.shape}")
print(student_performance.head())



Student performance aggregated: (32548, 6)
   id_student code_module code_presentation  total_weight  sum_weighted_score  \
0       11391         AAA             2013J         100.0              8240.0   
1       28400         AAA             2013J         100.0              6540.0   
2       30268         AAA             2013J         100.0                 NaN   
3       31604         AAA             2013J         100.0              7630.0   
4       32885         AAA             2013J         100.0              5500.0   

   tma_cma_weighted_score  
0                    82.4  
1                    65.4  
2                     NaN  
3                    76.3  
4                    55.0  


Missing Values as 0 for tma_cma_weighted_score. As the student might not have given the assessment or there was no tma_cma weight

In [86]:
print(f"\nScore statistics:")
print(student_performance['tma_cma_weighted_score'].describe())

before_fill = student_performance.isnull().sum()
print(f"\nMissing values in student performance:{before_fill}")
if before_fill.any():
    student_performance['tma_cma_weighted_score'] = student_performance['tma_cma_weighted_score'].fillna(0)
    print(f"Missing values after fill: {student_performance.isnull().sum()}")
    
# Reliability check: flag modules where weight coverage is low
low_coverage = student_performance[student_performance['total_weight'] < 50][['code_module', 'code_presentation', 'total_weight']].drop_duplicates()
if len(low_coverage) > 0:
    print("\n⚠ WARNING: Modules with low TMA/CMA weight coverage (<50%):")
    print(low_coverage)
else:
    print("\n✓ All modules have reasonable TMA/CMA weight coverage")


Score statistics:
count    23723.000000
mean        52.917182
std         29.740180
min          0.000000
25%         22.995000
50%         60.550000
75%         78.685000
max        100.000000
Name: tma_cma_weighted_score, dtype: float64

Missing values in student performance:id_student                   0
code_module                  0
code_presentation            0
total_weight                 0
sum_weighted_score        6713
tma_cma_weighted_score    8825
dtype: int64
Missing values after fill: id_student                   0
code_module                  0
code_presentation            0
total_weight                 0
sum_weighted_score        6713
tma_cma_weighted_score       0
dtype: int64

⚠ WARNING: Modules with low TMA/CMA weight coverage (<50%):
      code_module code_presentation  total_weight
30014         GGG             2013J           0.0
30966         GGG             2014B           0.0
31799         GGG             2014J           0.0


Drop Repeated Columns

In [87]:
col_to_remove = ['sum_weighted_score', 'total_weight']
if all(col in student_performance.columns.to_list() for col in col_to_remove):
    student_performance = student_performance.drop(columns=col_to_remove)
    print(f"\nDropped intermediate columns: {col_to_remove}")

student_performance.head()


Dropped intermediate columns: ['sum_weighted_score', 'total_weight']


,id_student,code_module,code_presentation,tma_cma_weighted_score
0,11391,AAA,2013J,82.4
1,28400,AAA,2013J,65.4
2,30268,AAA,2013J,0.0
3,31604,AAA,2013J,76.3
4,32885,AAA,2013J,55.0


### Merging Performance Score into Master Dataframe


In [88]:
# Master dataframe before performance merge
print(f"Master dataframe before performance merge: {master.shape}")


Master dataframe before performance merge: (32548, 19)


In [89]:
# Merge only the performance score to master dataframe
master = master.merge(
    student_performance[join_keys + ['tma_cma_weighted_score']],
    on=join_keys,
    how='left'
)

print(f"Master dataframe after performance merge: {master.shape}")
print(f"Performance score column present: {'tma_cma_weighted_score' in master.columns}")
print(f"Missing values in tma_cma_weighted_score: {master['tma_cma_weighted_score'].isnull().sum()}")


Master dataframe after performance merge: (32548, 20)
Performance score column present: True
Missing values in tma_cma_weighted_score: 0


## Step 3: Final Feature Engineering

Create derived features for modeling: withdrawal status, registration latency, and click density.


In [90]:
print(f"Shape of dataset before feature engineering: {master.shape}")

Shape of dataset before feature engineering: (32548, 20)


In [93]:
print(f"Columns before feature engineering: {master.columns.tolist()}")

Columns before feature engineering: ['code_module', 'code_presentation', 'id_student', 'gender', 'region', 'highest_education', 'imd_band', 'age_band', 'num_of_prev_attempts', 'studied_credits', 'disability', 'final_result', 'date_registration', 'total_clicks', 'avg_clicks_per_day', 'num_sites', 'num_unique_sites', 'module_presentation_length', 'total_unique_activities', 'tma_cma_weighted_score']


### Feature 1: Withdrawal Status


In [99]:
# Create binary withdrawal flag
is_withdrawn = (master["final_result"] == 'Withdrawn').astype(int)
# master['is_withdrawn'] = is_withdrawn

print("Withdrawal status distribution:")
print(is_withdrawn.value_counts())
print(f"Withdrawal rate: {is_withdrawn.mean()*100:.2f}%")

Withdrawal status distribution:
final_result
0    22431
1    10117
Name: count, dtype: int64
Withdrawal rate: 31.08%


## Step 4: Export & Verify

Save the final integrated master dataset and validate its structure.


### Saving Master Dataset


In [100]:
# Save master dataframe to CSV
output_path = MERGED_DIR / 'merged.csv'
master.to_csv(output_path, index=False)

print(f"Master dataset saved to: {output_path}")
print(f"File size: {output_path.stat().st_size / (1024**2):.2f} MB")

Master dataset saved to: ..\..\data\merged\merged.csv
File size: 3.94 MB


### Verification: Dataset Structure and Integrity


In [101]:
# Verify one row per student-module-presentation
uniqueness_check = master.groupby(join_keys).size()

print(f"Total rows in master dataset: {master.shape[0]}")
print(f"Total columns: {master.shape[1]}")
print(f"\nUnique student-module-presentation combinations: {len(uniqueness_check)}")
print(f"Max rows per combination: {uniqueness_check.max()}")
print(f"Min rows per combination: {uniqueness_check.min()}")

if uniqueness_check.max() == 1:
    print("\n✓ SUCCESS: One row per student-module-presentation!")
else:
    print(f"\n⚠ WARNING: Found {(uniqueness_check > 1).sum()} duplicate combinations")

Total rows in master dataset: 32548
Total columns: 20

Unique student-module-presentation combinations: 32548
Max rows per combination: 1
Min rows per combination: 1

✓ SUCCESS: One row per student-module-presentation!


In [102]:
# Display dataset summary
print("="*80)
print("MASTER STUDENT DATASET SUMMARY")
print("="*80)
print(f"\nShape: {master.shape[0]} rows × {master.shape[1]} columns")
print(f"\nFirst few rows:")
print(master.head())

print(f"\nColumn names and types:")
print(master.dtypes)

print(f"\nKey statistics:")
print(f"  - Modules: {master['code_module'].nunique()}")
print(f"  - Presentations: {master['code_presentation'].nunique()}")
print(f"  - Total students: {master['id_student'].nunique()}")
print(f"  - Average TMA/CMA score: {master['tma_cma_weighted_score'].mean():.2f}/100")
print(f"  - Average total clicks per student: {master['total_clicks'].mean():.2f}")

MASTER STUDENT DATASET SUMMARY

Shape: 32548 rows × 20 columns

First few rows:
  code_module code_presentation  id_student gender                region  \
0         AAA             2013J       11391      M   East Anglian Region   
1         AAA             2013J       28400      F              Scotland   
2         AAA             2013J       30268      F  North Western Region   
3         AAA             2013J       31604      F     South East Region   
4         AAA             2013J       32885      F  West Midlands Region   

       highest_education imd_band age_band  num_of_prev_attempts  \
0       HE Qualification  90-100%      55+                     0   
1       HE Qualification   20-30%    35-55                     0   
2  A Level or Equivalent   30-40%    35-55                     0   
3  A Level or Equivalent   50-60%    35-55                     0   
4     Lower Than A Level   50-60%     0-35                     0   

   studied_credits disability final_result  date_regis